# kvplan × vLLM: measure real KV-cache capacity

This notebook starts vLLM on whatever GPU Colab gives you. For each model that fits, it records how much
memory vLLM gives the KV cache and how many tokens that holds. It saves one JSON file per run. Back on your
laptop, `node validation/compare.ts` compares those measurements with kvplan's predictions.

## How to run it

1. **Open it in Colab:**
   <https://colab.research.google.com/github/Dubbus/kvplan/blob/main/notebooks/validate_vllm.ipynb>
2. **Pick a GPU:** *Runtime → Change runtime type → Hardware accelerator*. The free tier gives a **T4**.
   **L4** or **A100** need Colab Pro or compute units. The notebook detects the GPU and only runs models that fit.
   - T4 (16 GB, no bf16, no fp8): Qwen2.5 0.5B / 1.5B / 3B in fp16.
   - L4 / A100 / H100: those, plus Qwen2.5-7B, Mistral-7B and (with a token) Llama-3-8B. On L4/H100 it also runs an fp8 KV-cache variant.
3. **Optional, for Llama 3 (gated):** accept the license at
   <https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct>, then create a read token at
   <https://huggingface.co/settings/tokens>. In Colab, click the **🔑 Secrets** icon in the left sidebar, add a
   secret named `HF_TOKEN` with the token as its value, and turn on **Notebook access**. Without a token, the
   Llama runs are skipped and everything else still runs.
4. **Run it:** *Runtime → Run all*. The install takes about 3–5 minutes, and each model takes about 1–4 minutes.
   Don't restart the runtime after the install cell; the notebook never imports torch in this kernel, so no restart is needed.
5. **Get the results:** the last cell zips them and downloads `kvplan_results_<gpu>.zip`. Unzip it into the repo's
   `validation/results/` folder and run `node validation/compare.ts`.

If a run fails, the notebook records the error and the end of vLLM's log in that run's JSON, then moves on to the
next run. If *every* run fails at startup, try setting `EXTRA_ARGS = ["--enforce-eager"]` in the config cell and
run all again.

In [ ]:
# ---- Config ----
VLLM_VERSION = "0.19.0"   # pinned; see the note in the install cell
MAX_MODEL_LEN = 4096       # --max-model-len for every run
BLOCK_SIZE = 16            # --block-size
GPU_MEM_UTIL = 0.9         # --gpu-memory-utilization
EXTRA_ARGS = []            # e.g. ["--enforce-eager"] if startup fails on your GPU
TRY_FP8_ON_AMPERE = False  # fp8 KV cache runs by default only on compute capability >= 8.9 (L4, H100)
STARTUP_TIMEOUT_S = 900    # per run
RESULTS_DIR = "/content/kvplan/validation/results"

In [ ]:
# ---- GPU check (before installing anything) ----
import subprocess, sys
smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if smi.returncode != 0:
    raise SystemExit("No GPU found. Runtime -> Change runtime type -> pick T4 / L4 / A100, then Run all again.")
print(smi.stdout)
NVIDIA_SMI = smi.stdout

In [ ]:
# ---- Install vLLM ----
# Pinned to 0.19.0: it is reported working on the Colab 2026.04 runtime (T4, CUDA 12.8); newer default
# wheels target CUDA 12.9/13.0 and fail on Colab with "libcudart.so.13: cannot open shared object file".
# vLLM requires compute capability >= 7.5, so the T4 (Turing, 7.5) is the oldest GPU it supports.
# The pip wheels bring their own torch build; this kernel never imports torch, so no restart is needed.
!pip install -q "vllm=={VLLM_VERSION}"
!vllm --version

In [ ]:
# ---- Record the environment (in a subprocess, so this kernel never holds GPU memory) ----
import json
probe = r"""
import json, torch, vllm
free, total = torch.cuda.mem_get_info()
p = torch.cuda.get_device_properties(0)
print(json.dumps({
    "name": torch.cuda.get_device_name(0),
    "computeCapability": f"{p.major}.{p.minor}",
    "totalBytes": total, "freeBytes": free,
    "torchVersion": torch.__version__, "torchCuda": torch.version.cuda,
    "vllmVersion": vllm.__version__,
}))
"""
out = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
if out.returncode != 0:
    print(out.stderr[-3000:])
    raise SystemExit("Probe failed: torch/vLLM can't see the GPU. See the error above.")
ENV = json.loads(out.stdout.strip().splitlines()[-1])
CC = tuple(int(x) for x in ENV["computeCapability"].split("."))

def gpu_slug(name, total):
    if "A100" in name: return "A100-80GB" if total > 60 * 2**30 else "A100-40GB"
    for key, slug in [("H100", "H100-80GB"), ("A10G", "A10G"), ("L4", "L4"), ("T4", "T4"), ("4090", "RTX4090")]:
        if key in name: return slug
    return "".join(c if c.isalnum() else "-" for c in name).strip("-")

GPU_SLUG = gpu_slug(ENV["name"], ENV["totalBytes"])
WEIGHT_DTYPE = "bfloat16" if CC >= (8, 0) else "half"   # T4 has no bf16
KV_DTYPES = ["auto"] + (["fp8"] if CC >= (8, 9) or (TRY_FP8_ON_AMPERE and CC >= (8, 0)) else [])
print(json.dumps(ENV, indent=2))
print(f"GPU slug: {GPU_SLUG}; weights: {WEIGHT_DTYPE}; KV cache dtypes: {KV_DTYPES}")
print(f"total = {ENV['totalBytes'] / 2**30:.2f} GiB")

In [ ]:
# ---- Hugging Face token (optional; only needed for gated Llama models) ----
import os
HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or HF_TOKEN
except Exception as e:  # secret missing, or notebook access not granted
    print(f"No HF_TOKEN secret ({type(e).__name__}); gated models will be skipped.")
print("HF token:", "found" if HF_TOKEN else "not set")

In [ ]:
# ---- Models: mirrors kvplan's src/presets.ts (verified against each repo's config.json) ----
MODELS = [
    # preset key,     HF repo,                               params,         layers, kv_heads, head_dim, gated
    ("qwen2.5_0.5b", "Qwen/Qwen2.5-0.5B-Instruct",            494_032_768,   24, 2, 64,  False),
    ("qwen2.5_1.5b", "Qwen/Qwen2.5-1.5B-Instruct",          1_543_714_304,   28, 2, 128, False),
    ("qwen2.5_3b",   "Qwen/Qwen2.5-3B-Instruct",            3_085_938_688,   36, 2, 128, False),
    ("qwen2.5_7b",   "Qwen/Qwen2.5-7B-Instruct",            7_615_616_512,   28, 4, 128, False),
    ("mistral_7b",   "mistralai/Mistral-7B-Instruct-v0.3",  7_248_023_552,   32, 8, 128, False),
    ("llama3_8b",    "meta-llama/Meta-Llama-3-8B-Instruct", 8_030_261_248,   32, 8, 128, True),
]
GiB = 2**30
budget = ENV["totalBytes"] * GPU_MEM_UTIL
PLAN = []  # (preset, hf_id, kv_dtype)
for preset, hf_id, params, layers, kvh, hd, gated in MODELS:
    weights = params * 2
    one_seq = 2 * layers * kvh * hd * 2 * MAX_MODEL_LEN
    # Generous margin (3 GiB) for activations / CUDA graphs, so we only try runs that should start.
    fits = weights + 3 * GiB + one_seq < budget
    why = "" if fits else f"too big: {weights / GiB:.1f} GiB weights vs {budget / GiB:.1f} GiB budget"
    if fits and gated and not HF_TOKEN:
        fits, why = False, "gated, no HF_TOKEN"
    print(f"{'RUN ' if fits else 'skip'} {preset:14s} {why}")
    if fits:
        PLAN += [(preset, hf_id, kv) for kv in KV_DTYPES]
print(f"\n{len(PLAN)} runs planned")

In [ ]:
# ---- Runner: one vLLM server per run, in its own process group; killed afterwards ----
import os, re, signal, threading, time

PATTERNS = {  # keep in sync with parseVllmLog in validation/compare.ts
    "availableKvGiB": re.compile(r"Available KV cache memory:\s*([\d.,]+)\s*GiB", re.I),
    "v0AvailableKvGiB": re.compile(r"reserved for KV Cache is\s*([\d.,]+)\s*GiB", re.I),
    "kvCacheTokens": re.compile(r"GPU KV cache size:\s*([\d,]+)\s*tokens", re.I),
    "numGpuBlocks": re.compile(r"#\s*GPU blocks:\s*([\d,]+)|num_gpu_blocks\s*[=:]\s*([\d,]+)", re.I),
    "modelLoadingGiB": re.compile(r"Model loading took\s*([\d.,]+)\s*GiB", re.I),
    "maxConcurrency": re.compile(r"Maximum concurrency for [\d,]+ tokens per request:\s*([\d.,]+)x", re.I),
}
KEEP = re.compile(r"KV cache|GPU blocks|Model loading took|memory profiling|CUDA graph|concurrency|"
                  r"Using .*backend|attention backend|max_num_batched_tokens|out of memory|Error", re.I)
READY = re.compile(r"Application startup complete|Uvicorn running on", re.I)

def gpu_used_mib():
    q = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                       capture_output=True, text=True)
    return int(q.stdout.split()[0]) if q.returncode == 0 else -1

def wait_gpu_free(timeout=180):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if 0 <= gpu_used_mib() < 1024:
            return True
        time.sleep(3)
    return False

def parse(lines):
    m = {}
    for line in lines:
        for key, rx in PATTERNS.items():
            hit = rx.search(line)
            if hit:
                val = next(g for g in hit.groups() if g is not None)
                m[key] = float(val.replace(",", ""))
    if "availableKvGiB" not in m and "v0AvailableKvGiB" in m:
        m["availableKvGiB"] = m["v0AvailableKvGiB"]
    m.pop("v0AvailableKvGiB", None)
    if "availableKvGiB" in m:
        m["availableKvBytes"] = m["availableKvGiB"] * GiB
    if "modelLoadingGiB" in m:
        m["modelLoadingBytes"] = m["modelLoadingGiB"] * GiB
    for k in ("kvCacheTokens", "numGpuBlocks"):
        if k in m: m[k] = int(m[k])
    return m

def run_one(preset, hf_id, kv_dtype, port=8011):
    flags = {"maxModelLen": MAX_MODEL_LEN, "blockSize": BLOCK_SIZE, "gpuMemoryUtilization": GPU_MEM_UTIL,
             "kvCacheDtype": kv_dtype, "dtype": WEIGHT_DTYPE, "extraArgs": EXTRA_ARGS}
    cmd = ["vllm", "serve", hf_id, "--port", str(port),
           "--max-model-len", str(MAX_MODEL_LEN), "--block-size", str(BLOCK_SIZE),
           "--gpu-memory-utilization", str(GPU_MEM_UTIL), "--kv-cache-dtype", kv_dtype,
           "--dtype", WEIGHT_DTYPE, *EXTRA_ARGS]
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    if HF_TOKEN: env["HF_TOKEN"] = HF_TOKEN
    print("$ " + " ".join(cmd))
    free_before = ENV["totalBytes"] - gpu_used_mib() * 2**20
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            env=env, start_new_session=True)
    lines, ready = [], threading.Event()
    def pump():
        for line in proc.stdout:
            lines.append(line.rstrip("\n"))
            if KEEP.search(line): print("   " + line.rstrip()[-200:])
            if READY.search(line): ready.set()
    threading.Thread(target=pump, daemon=True).start()
    t0, status, error = time.time(), "ok", None
    while not ready.is_set():
        if proc.poll() is not None:
            time.sleep(1)  # let the pump drain
            status, error = "failed", f"vLLM exited with code {proc.returncode} before startup completed"
            break
        if time.time() - t0 > STARTUP_TIMEOUT_S:
            status, error = "failed", f"startup timed out after {STARTUP_TIMEOUT_S}s"
            break
        time.sleep(2)
    startup_s = time.time() - t0
    # Kill the whole process group (API server + EngineCore), politely then firmly. Keep going
    # until every process in the group is gone: the server exiting doesn't mean its children did.
    def group_alive():
        proc.poll()  # reap the leader so a zombie doesn't count as alive
        try: os.killpg(proc.pid, 0); return True
        except ProcessLookupError: return False
        except PermissionError: return True
    for sig, grace in ((signal.SIGINT, 20), (signal.SIGTERM, 15), (signal.SIGKILL, 10)):
        if not group_alive(): break
        try: os.killpg(proc.pid, sig)
        except ProcessLookupError: break
        deadline = time.time() + grace
        while group_alive() and time.time() < deadline:
            time.sleep(0.5)
    if not wait_gpu_free():
        print(f"   warning: GPU still shows {gpu_used_mib()} MiB used after shutdown")
    measured = parse(lines)
    if status == "ok" and "kvCacheTokens" not in measured and "numGpuBlocks" not in measured:
        status, error = "failed", "server started but no KV cache size found in the log"
    result = {
        "schema": 1, "status": status, "error": error,
        "gpu": {"name": ENV["name"], "slug": GPU_SLUG, "totalBytes": ENV["totalBytes"],
                "freeBytesBefore": free_before, "computeCapability": ENV["computeCapability"],
                "nvidiaSmi": NVIDIA_SMI},
        "vllmVersion": ENV["vllmVersion"], "torchVersion": ENV["torchVersion"], "torchCuda": ENV["torchCuda"],
        "model": {"preset": preset, "hfId": hf_id},
        "flags": flags, "command": cmd, "startupSeconds": round(startup_s, 1),
        "measured": measured,
        "rawLogLines": [l for l in lines if KEEP.search(l)],
        "logTail": lines[-80:],
        "recordedAt": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    path = f"{RESULTS_DIR}/{GPU_SLUG}_{preset}_{kv_dtype}.json"
    with open(path, "w") as f:
        json.dump(result, f, indent=2)
    tag = "OK " if status == "ok" else "FAIL"
    print(f"{tag} {preset} kv={kv_dtype}: {measured.get('kvCacheTokens', '?')} tokens, "
          f"{measured.get('availableKvGiB', '?')} GiB KV, {startup_s:.0f}s  -> {path}"
          + (f"\n     {error}" if error else ""))
    return result

In [ ]:
# ---- Run everything ----
os.makedirs(RESULTS_DIR, exist_ok=True)
RESULTS = []
for preset, hf_id, kv_dtype in PLAN:
    print(f"\n=== {GPU_SLUG} / {preset} / kv {kv_dtype} ===")
    try:
        RESULTS.append(run_one(preset, hf_id, kv_dtype))
    except Exception as e:  # never let one run stop the rest
        print(f"FAIL {preset} kv={kv_dtype}: notebook error {e!r}")

In [ ]:
# ---- Summary + download ----
import shutil
print(f"{'model':14s} {'kv':5s} {'status':7s} {'KV GiB':>8s} {'KV tokens':>11s}")
for r in RESULTS:
    m = r["measured"]
    print(f"{r['model']['preset']:14s} {r['flags']['kvCacheDtype']:5s} {r['status']:7s} "
          f"{m.get('availableKvGiB', float('nan')):8.2f} {m.get('kvCacheTokens', 0):11,d}")
zip_path = shutil.make_archive(f"/content/kvplan_results_{GPU_SLUG}", "zip", RESULTS_DIR)
print(f"\nWrote {zip_path}. Unzip it into the repo's validation/results/ and run: node validation/compare.ts")
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print(f"Auto-download unavailable ({type(e).__name__}); use the Files panel on the left.")